# 05 · Successions i criteri de Cauchy

**Matemàtiques · 1r de batxillerat**  
**Autoria: Dr. Lacasa-Cazcarra · Any 2026**

**Objectiu.** Distingir convergència, acotació i condició de Cauchy, i precisar els quantificadors.

**Abans de començar:** successions elementals i quadern 01  
**Temps orientatiu:** 2 sessions. Hi ha activitats bàsiques i ampliacions opcionals.

**Funcionament.** Executa les cel·les en ordre, des del començament. Primer fes una predicció,
després experimenta i escriu una justificació. Els controls són opcionals: també pots
modificar els arguments de les funcions i executar-les. Les figures representen mostres
finites; les propietats infinites necessiten una demostració.

**Procedència.** Quadern nou del recorregut de fonaments.


In [ ]:
import math
from fractions import Fraction
from decimal import Decimal, localcontext
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "font.size": 11, "figure.dpi": 100})

def explora(funcio, **rangs):
    """Controls opcionals: la mateixa funció també es pot cridar directament."""
    try:
        import ipywidgets as widgets
    except ImportError:
        print("Sense controls: executem els paràmetres per defecte. Pots canviar-los a la crida.")
        return funcio()
    panell = widgets.interactive(funcio, **rangs)
    display(panell)
    return panell


## 1. Apropar-se a un límit o apropar-se entre si
$a_n\to L$ significa:
$$\forall\varepsilon>0\ \exists N\ \forall n\ge N:\ |a_n-L|<\varepsilon.$$
Una successió és **de Cauchy** si:
$$\forall\varepsilon>0\ \exists N\ \forall m,n\ge N:\ |a_m-a_n|<\varepsilon.$$
La segona definició no necessita conèixer L. L'índex N pot dependre d'ε,
però ha de servir per a **tots dos índexs** m i n de la cua infinita.

Prediu què passarà amb $1/n$, $(-1)^n$ i les sumes harmòniques.


In [ ]:
def valors_successio(tipus, maxim):
    n = np.arange(1, maxim+1)
    if tipus == "inversa":
        return 1/n
    if tipus == "alternant":
        return (-1.0)**n
    if tipus == "harmonica":
        return np.cumsum(1/n)
    raise ValueError("Successió desconeguda.")

def laboratori_cauchy(tipus="inversa", N=10, finestra=60, epsilon=0.2):
    valors = valors_successio(tipus, N+finestra-1)
    cua = valors[N-1:]
    diametre = float(np.max(cua)-np.min(cua))
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(12, 4))
    ax.plot(np.arange(N, N+finestra), cua, ".", color="#2563eb")
    ax.axhspan(cua[0]-epsilon, cua[0]+epsilon, alpha=.15, color="#16a34a")
    ax.set(xlabel="n", ylabel="aₙ", title="Finestra finita de la cua")
    distancies = np.abs(cua[:, None]-cua[None, :])
    im = bx.imshow(distancies, origin="lower", aspect="auto", cmap="viridis")
    bx.set(xlabel="Índex dins la finestra", ylabel="Índex dins la finestra", title="Distàncies |aₘ − aₙ|")
    fig.colorbar(im, ax=bx); plt.tight_layout(); plt.show()
    print(f"Diàmetre mostrat = {diametre:.6g}; ε = {epsilon}")
    print("Totes les parelles MOSTRADES compleixen la desigualtat:", diametre < epsilon)
    print("Una comprovació finita no certifica una cua infinita.")
panell = explora(laboratori_cauchy, tipus=["inversa", "alternant", "harmonica"],
                 N=(1, 300, 1), finestra=(10, 150, 10), epsilon=(.01, 2.1, .01))


## 2. Tres justificacions, no només tres gràfiques
**$1/n$:** si $m,n\ge N$, $|1/m-1/n|<1/N$. Escollint $N>1/\varepsilon$,
la successió és de Cauchy i convergeix a 0.

**$(-1)^n$:** per a qualsevol N hi ha índexs parells i senars més grans que N,
amb distància 2. Prenent ε=1 refutem la condició de Cauchy. Estar acotada no basta.

**Harmònica:** els salts consecutius $H_{n+1}-H_n=1/(n+1)$ tendeixen a zero, però
$$H_{2n}-H_n=\sum_{k=n+1}^{2n}\frac1k\ge n\frac1{2n}=\frac12.$$
Per a ε=1/2, cap N serveix. Per això «els termes consecutius s'apropen» no és el criteri de Cauchy.


In [ ]:
for n in (10, 100, 1000, 10000):
    bloc = np.sum(1 / np.arange(n+1, 2*n+1))
    print(f"n={n:5}: salt consecutiu ≈ {1/(n+1):.7f}; H₂ₙ−Hₙ ≈ {bloc:.7f}")


In [ ]:
def intervals_arrel2(passos=12):
    """Extrems racionals exactes. No calcula sqrt(2)."""
    if not isinstance(passos, int) or passos < 0:
        raise ValueError("El nombre de passos ha de ser un enter no negatiu.")
    a, b = Fraction(1), Fraction(2)
    intervals = [(a, b)]
    for _ in range(passos):
        m = (a + b) / 2
        if m * m < 2:
            a = m
        else:
            b = m
        intervals.append((a, b))
    return intervals


In [ ]:
intervals = intervals_arrel2(20)
aproximacions = [(a+b)/2 for a, b in intervals]
for n in (2, 5, 10, 20):
    a, b = intervals[n]
    print(f"n={n}: terme racional {aproximacions[n]}; amplada exacta {b-a}")


## 3. De Cauchy en ℚ, però sense límit en ℚ
Els punts mitjans $c_n$ dels intervals de bisecció són racionals.
Si $m,n\ge N$, tots dos són dins $[a_N,b_N]$, de manera que
$|c_m-c_n|\le2^{-N}$. Escollint $2^{-N}<\varepsilon$, demostrem que són de Cauchy.

Si tinguessin un límit racional r, les desigualtats $a_n^2<2<b_n^2$ i l'amplada
que tendeix a zero implicarien $r^2=2$, impossible pel quadern 01.
En ℝ sí que convergeixen: aquest és el paper de la completesa.

**Tota successió convergent és de Cauchy:** si a partir de N tots els termes són
a menys d'ε/2 de L, la desigualtat triangular dona
$|a_m-a_n|\le|a_m-L|+|L-a_n|<\varepsilon$.

## Activitats
1. Escriu un N que serveixi per a $a_n=1/n$ quan ε=0,001.
2. A la successió harmònica, augmenta N mantenint una finestra curta. Per què la
   pantalla pot suggerir una conclusió falsa? Torna a comparar n i 2n.
3. Explica amb paraules pròpies la diferència entre «per a tot» i «existeix» en la definició.
4. **Ampliació:** demostra que tota successió de Cauchy és acotada.

<details><summary>Pistes</summary>

N=1001 és suficient. Per a l'acotació, usa ε=1: tota la cua queda a distància
menor que 1 d'un terme fix. Els termes anteriors són un conjunt finit, també acotat.
Una finestra de longitud fixa pot ocultar distàncies grans entre índexs més separats.
</details>


**Navegació:** [Anterior](04_Talladures_Dedekind.ipynb) · [Índex i guia](README.md) · [Següent](06_Completesa_R.ipynb)
